# Production Scheduling & Monitoring — TPC-H Sales & Supply Chain Job

## How the pipeline is scheduled in production

The entire medallion pipeline is orchestrated by a single Lakeflow Job — **TPC-H Sales & Supply Chain Job** — composed of **four tasks chained by dependencies**, so each stage only starts when the previous one succeeds:

```
[Task 1: Bronze Ingestion]
        |
        v
[Task 2: Silver Cleaning (8 tables, parent-before-child)]
        |
        v
[Task 3: Gold Layer (dims + fact_sales)]
        |
        v
[Task 4: Data Quality Gate]
```

| Task | Stage | Depends on | What it does |
| --- | --- | --- | --- |
| 1. Bronze Ingestion | `01_ingestion` | — | Reads all 8 `samples.tpch` source tables and **overwrites** `dev.bronze.*` with lineage metadata (`_ingestion_timestamp`, `_source_file_path`, `_pipeline_run_id`). Overwrite mode makes the run idempotent and re-runnable. |
| 2. Silver Cleaning | 8 × `02_<table>_cleaning` | Task 1 | Cleans and validates each table (PK/FK checks, value-range rules), writes `dev.silver.<t>` plus a `_quarantine` table, and asserts **Bronze = Silver + Quarantine** at the end of every notebook. Runs in parent-before-child order: `region → nation → part / supplier / customer → orders → partsupp / lineitem` so FK joins always find an existing parent table. |
| 3. Gold Layer | `03_fact_and_dim` | Task 2 | Rebuilds `dim_geography`, `dim_customer`, `dim_product`, `dim_supplier` and `fact_sales`, ending with an independent reconciliation against Silver (row counts and net revenue must match). |
| 4. Data Quality Gate | `04_data_quality` | Task 3 | Runs the DQ-01 … DQ-06 suite (source presence, PK uniqueness, surrogate-key completeness, business rules, Gold↔Silver reconciliation, rejected-record accounting) and **raises a RuntimeError on any CRITICAL failure** — so the job fails loudly instead of silently publishing bad Gold data. |

### Scheduling configuration

* **Trigger:** cron schedule (e.g. `0 2 * * *` — daily at 02:00), since the batch refresh (overwrite) makes the whole pipeline deterministic.
* **Parameters:** the `CATALOG` widget value is promoted to a **job parameter** (e.g. `CATALOG=dev`), so dev/prod catalogs can be swapped without editing any notebook code.
* **Compute:** serverless compute — no cluster to warm up or maintain, and each task scales independently.
* **Idempotency:** every layer writes with overwrite, so a **repair run** (re-running only failed tasks) is always safe — no partial-state cleanup needed.

## How the pipeline is monitored in production

* **Job run UI:** per-task status, duration, and start/end times for every run; each task links directly to its notebook run, so triage starts one click from the failure.
* **Notifications:** email / webhook / Slack alerts configured on **job failure, task failure, and timeout**, with "alert on first failure only" to avoid alert storms during retries.
* **Retries & repair:** transient failures (e.g. temporary source unavailability) are handled by automatic task retries; persistent failures are fixed and re-run via **repair run**, which only executes failed tasks and their dependents.
* **System tables:** `system.lakeflow.job_run_timeline` gives account-wide run history for trend analysis (success rate, duration drift), and `system.billing.usage` tracks the cost of each run — useful for spotting a task that suddenly scans more data than usual.
* **Data-quality monitoring:** the persisted `dq_report_*` tables and quarantine-table row counts are the leading indicator of **source drift** — a spike in quarantined rows or a reconciliation delta > 0.01% means the source data changed contract, and the DQ gate blocks the pipeline before it reaches Gold.
* **SLA tracking:** success rate and end-to-end duration are tracked per run against a baseline; a scheduled **SQL alert** on the job-run history notifies the team if the daily run misses its SLA window (e.g. not finished by 04:00).

### Failure handling summary

| Failure | Response |
| --- | --- |
| Transient infrastructure error | Automatic retry, then notification |
| DQ gate CRITICAL failure (Task 4) | Job marked FAILED, Gold data from the last good run stays in place, on-call triages via the DQ report tables |
| Source schema / contract change | Quarantine spike caught by DQ checks; notebook fix, then repair run from Task 2 |

**Bottom line:** the four-task dependency chain (Bronze → Silver → Gold → DQ gate) means the pipeline either completes fully and passes all data-quality checks, or fails fast at the exact stage where the problem occurred — with idempotent overwrites making every rerun and repair safe.